# Input Corpus Shuffle

Shuffle input corpuses at the sample-record level before preprocessing.

This notebook is UI-only and calls Python modules in `rb_pipeline/corpus_shuffle`.


In [1]:
from pathlib import Path
import sys

def find_repo_root(start: Path | None = None) -> Path:
    candidate = (start or Path.cwd()).resolve()
    if candidate.is_file():
        candidate = candidate.parent

    for current in (candidate, *candidate.parents):
        if (current / 'input-images').exists() and (current / 'rb_pipeline_v3').exists():
            return current

    raise RuntimeError('Could not locate repository root with input-images/ and rb_pipeline_v3/.')

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

REPO_ROOT


PosixPath('/home/mitch/development/raccoon-ball/synthetic-data-processing')

In [2]:
import html

import ipywidgets as widgets
from IPython.display import HTML, clear_output, display

from rb_pipeline.corpus_shuffle import (
    default_destination_path,
    discover_corpuses,
    parse_seed,
    shuffle_corpus,
)

INPUT_ROOT = REPO_ROOT / 'input-images'
_summaries = []

title = widgets.HTML('<b>Input Corpus Shuffle (Preprocessing)</b>')
summary_html = widgets.HTML()
seed_error_html = widgets.HTML()

corpus_dropdown = widgets.Dropdown(
    description='Corpus:',
    options=[],
    layout=widgets.Layout(width='98%'),
)
seed_text = widgets.Text(
    description='Seed:',
    value='',
    placeholder='Required integer seed',
)
refresh_button = widgets.Button(description='Refresh Corpuses')
execute_button = widgets.Button(
    description='Execute Shuffle',
    button_style='primary',
    disabled=True,
)
status_output = widgets.Output(layout=widgets.Layout(border='1px solid #ccc', padding='8px'))


def _seed_error() -> str | None:
    try:
        parse_seed(seed_text.value)
    except Exception as exc:
        return str(exc)
    return None


def _selected_corpus_path() -> Path | None:
    value = corpus_dropdown.value
    if not value:
        return None
    return Path(value)


def _render_summary_table(summaries) -> str:
    if not summaries:
        return '<i>No corpuses found under input-images.</i>'

    header = (
        '<tr>'
        '<th align="left">Corpus</th>'
        '<th align="left">Path</th>'
        '<th align="left">samples.csv</th>'
        '<th align="left">run.json</th>'
        '<th align="right">Rows</th>'
        '<th align="right">Images Found</th>'
        '<th align="right">Images Missing</th>'
        '<th align="left">Validates</th>'
        '<th align="left">Message</th>'
        '</tr>'
    )

    rows = []
    for summary in summaries:
        rows.append(
            '<tr>'
            f'<td>{html.escape(summary.name)}</td>'
            f'<td><code>{html.escape(str(summary.path))}</code></td>'
            f'<td>{"yes" if summary.has_samples_csv else "no"}</td>'
            f'<td>{"yes" if summary.has_run_json else "no"}</td>'
            f'<td align="right">{"" if summary.samples_row_count is None else summary.samples_row_count}</td>'
            f'<td align="right">{"" if summary.referenced_images_found is None else summary.referenced_images_found}</td>'
            f'<td align="right">{"" if summary.referenced_images_missing is None else summary.referenced_images_missing}</td>'
            f'<td>{"yes" if summary.validates_cleanly else "no"}</td>'
            f'<td>{html.escape(summary.validation_message)}</td>'
            '</tr>'
        )

    return (
        '<div style="max-height: 280px; overflow-y: auto; border: 1px solid #ddd; padding: 6px;">'
        '<table style="border-collapse: collapse; width: 100%;">'
        f'{header}{"".join(rows)}'
        '</table>'
        '</div>'
    )


def _update_execute_state(*_):
    seed_error = _seed_error()
    selected = _selected_corpus_path()

    execute_button.disabled = selected is None or seed_error is not None
    if seed_error:
        seed_error_html.value = f'<span style="color: #b00020;">Seed error: {html.escape(seed_error)}</span>'
    else:
        seed_error_html.value = ''


def _refresh_corpuses(*_):
    global _summaries
    _summaries = discover_corpuses(INPUT_ROOT)
    summary_html.value = _render_summary_table(_summaries)

    selectable = [summary for summary in _summaries if summary.selectable]
    options = []
    for summary in selectable:
        row_count = summary.samples_row_count if summary.samples_row_count is not None else 'n/a'
        status = 'valid' if summary.validates_cleanly else 'invalid'
        label = f'{summary.name} | rows={row_count} | {status}'
        options.append((label, str(summary.path)))

    corpus_dropdown.options = options
    if options:
        corpus_dropdown.value = options[0][1]
    else:
        corpus_dropdown.value = None

    _update_execute_state()


def _run_shuffle(_):
    with status_output:
        clear_output(wait=True)

        selected_corpus = _selected_corpus_path()
        if selected_corpus is None:
            print('Validation error: select one corpus before running.')
            return

        try:
            seed = parse_seed(seed_text.value)
        except Exception as exc:
            print(f'Validation error: {exc}')
            return

        destination = default_destination_path(selected_corpus)
        print(f'Selected corpus: {selected_corpus}')
        print(f'Selected seed: {seed}')
        print(f'Destination path: {destination}')

        try:
            result = shuffle_corpus(
                selected_corpus,
                seed=seed,
                destination_path=destination,
            )
        except Exception as exc:
            print(f'Shuffle failed: {exc}')
            return

        print('Shuffle completed successfully.')
        print(f'Output corpus: {result.destination_corpus_path}')
        print(f'Output rows: {result.output_row_count}')
        print(f'Excluded capture_success=false rows: {result.excluded_capture_failed_rows}')


refresh_button.on_click(_refresh_corpuses)
execute_button.on_click(_run_shuffle)
corpus_dropdown.observe(_update_execute_state, names='value')
seed_text.observe(_update_execute_state, names='value')

display(
    widgets.VBox(
        [
            title,
            widgets.HBox([refresh_button]),
            corpus_dropdown,
            seed_text,
            seed_error_html,
            execute_button,
            widgets.HTML('<b>Discovered Corpuses</b>'),
            summary_html,
            widgets.HTML('<b>Status</b>'),
            status_output,
        ]
    )
)

_refresh_corpuses()
